# 07 — Gold: Resumo de Data Quality de `ecommerce_itens_pedido` e `ecommerce_rastreamento_entregas`

Este notebook gera as tabelas Gold de qualidade das **duas tabelas desta dupla** e grava em dois destinos: Delta no ADLS e Azure SQL Server (consumido pelo Looker). Ele não cria regras novas; agrega o que já foi medido no `dq_monitoring_logs` (notebook 06) e na Silver (notebook 05).

## Regras/KPIs gerados por este notebook

Para cada uma das duas tabelas:

- **Resumo por regra** — 1 linha por (`tabela`, `regra`, `_data`), com `qtd_falhas`, `qtd_total` e `pct_falha` (% de registros que falharam a regra no dia). Fonte: `dq_monitoring_logs`. Inclui as 10 regras, inclusive as agregadas por pedido.
- **Resumo por tabela** — 1 linha por (`_data`, `_hora`), com `qtd_limpos`, `qtd_total` e `pct_limpo` (% de registros que passaram em **todas** as regras por linha ao mesmo tempo). Fonte: Silver, linha a linha. As regras agregadas por pedido não entram aqui porque não existem por linha.

`_data` e `_hora` vêm do `arquivo_origem` / `bronze_source_file` (`real-time-data/AAAA/MM/DD/HHMMSS/...`), ou seja, do momento do micro-lote, como acordado pela squad.

## Destinos

| Tabela | Delta (container `squad1`) | Azure SQL |
|---|---|---|
| `ecommerce_itens_pedido` | `gold/ecommerce_itens_pedido/dq_resumo_por_regra` e `.../dq_resumo_por_tabela` | `squad1.gold_dq_resumo_por_regra_itens_pedido` e `squad1.gold_dq_resumo_por_tabela_itens_pedido` |
| `ecommerce_rastreamento_entregas` | `gold/ecommerce_rastreamento_entregas/dq_resumo_por_regra` e `.../dq_resumo_por_tabela` | `squad1.gold_dq_resumo_por_regra_rastreamento_entregas` e `squad1.gold_dq_resumo_por_tabela_rastreamento_entregas` |

Os nomes das colunas seguem as outras Golds da squad (`_data`, `_hora`, `qtd_*`, `pct_*`) para permitir a união no Looker. Cada execução recalcula tudo e grava com `overwrite`, mas **só nos caminhos e tabelas desta dupla**.

In [0]:
# ============================================================
# CÉLULA 1 — SETUP (ADLS + SQL SERVER)
# ============================================================

from pyspark.sql import functions as F
from functools import reduce

# --- Credenciais ADLS ---
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

# --- Credenciais SQL Server ---
sql_host = dbutils.secrets.get(scope="internship-squad1", key="sql-host")
sql_database = dbutils.secrets.get(scope="internship-squad1", key="sql-database")
sql_user = dbutils.secrets.get(scope="internship-squad1", key="sql-username")
sql_password = dbutils.secrets.get(scope="internship-squad1", key="sql-password")

base_squad1 = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net"
dq_logs_path = f"{base_squad1}/dq_monitoring_logs"

# Configuração por tabela: Silver de origem, regras por linha (True = passou) e sufixo do nome no SQL
TABELAS = {
    "ecommerce_itens_pedido": {
        "silver": f"{base_squad1}/dupla1/silver/ecommerce_itens_pedido",
        "regras_linha": ["tecnica_1_pk_valida", "tecnica_2_fk_pedido_valido", "tecnica_3_fk_sku_valido",
                         "tecnica_4_quantidade_valida", "tecnica_5_preco_valido",
                         "negocio_6_desconto_nao_maior_preco", "negocio_8_desconto_nao_negativo",
                         "negocio_9_desconto_dentro_limite"],
        "sufixo_sql": "itens_pedido",
    },
    "ecommerce_rastreamento_entregas": {
        "silver": f"{base_squad1}/dupla1/silver/ecommerce_rastreamento_entregas",
        "regras_linha": ["tecnica_1_pk_valida", "tecnica_2_status_valido", "tecnica_3_fk_pedido_valido",
                         "tecnica_4_data_valida", "tecnica_5_codigo_rastreio_valido",
                         "negocio_6_ordem_cronologica"],
        "sufixo_sql": "rastreamento_entregas",
    },
}

def gold_path(tabela, nome):
    return f"{base_squad1}/gold/{tabela}/{nome}"

def data_do_lote(col):
    """'real-time-data/2026/10/15/105602/...' -> '2026-10-15'"""
    return F.regexp_replace(F.regexp_extract(col, r"real-time-data/(\d{4}/\d{2}/\d{2})/", 1), "/", "-")

def hora_do_lote(col):
    """'real-time-data/2026/10/15/105602/...' -> '10'"""
    return F.regexp_extract(col, r"real-time-data/\d{4}/\d{2}/\d{2}/(\d{2})\d{4}/", 1)

print("Setup concluído (ADLS + SQL Server).")
for t in TABELAS:
    print(f"  {t}: {gold_path(t, 'dq_resumo_por_regra')} | {gold_path(t, 'dq_resumo_por_tabela')}")

## Etapa 1 — Setup (ADLS + SQL Server)

Credenciais via Secret Scope (ADLS e SQL Server) e a configuração das duas tabelas da dupla: caminho da Silver, lista de regras **por linha** (colunas booleanas da Silver, `True` = passou) e o sufixo usado no nome das tabelas do SQL.

As funções `data_do_lote` e `hora_do_lote` extraem `_data` (`AAAA-MM-DD`) e `_hora` (`HH`) do caminho do micro-lote. Os caminhos Delta ficam em `squad1/gold/<tabela>/`, separados das Golds das outras duplas.

In [0]:
# ============================================================
# CÉLULA 2 — LER O LOG (SÓ AS TABELAS DESTA DUPLA)
# ============================================================

df_logs_bruto = (spark.read.format("delta").options(**adls_options).load(dq_logs_path)
    .filter(F.col("tabela").isin(list(TABELAS))))

# Linhas antigas, gravadas antes da correção do notebook 06, não têm arquivo_origem e ficam de fora
legado = df_logs_bruto.filter(F.col("arquivo_origem").isNull()).count()

df_logs = (df_logs_bruto
    .filter(F.col("arquivo_origem").isNotNull())
    .withColumn("_data", data_do_lote(F.col("arquivo_origem")))
    # Normaliza a severidade por segurança (Crítica/Critica -> Critica)
    .withColumn("severidade", F.when(F.lower(F.translate("severidade", "íÍ", "iI")) == "critica", "Critica")
                               .otherwise(F.col("severidade"))))

sem_data = df_logs.filter(F.col("_data") == "").count()
assert sem_data == 0, f"{sem_data} linhas do log com arquivo_origem fora do padrão real-time-data/AAAA/MM/DD/..."

print(f"Linhas do log usadas: {df_logs.count()}")
print(f"Linhas antigas ignoradas (sem arquivo_origem): {legado}")

## Etapa 2 — Leitura do log

Lê o `dq_monitoring_logs` filtrando **apenas as duas tabelas desta dupla**. As linhas gravadas pela versão antiga do notebook 06 (sem `arquivo_origem`) são ignoradas e contadas, para dar visibilidade até que sejam limpas em acordo com a squad. Como o log agora não duplica combinações de tabela + regra + arquivo, não é preciso deduplicar execuções: todos os micro-lotes entram na conta.

In [0]:
# ============================================================
# CÉLULA 3 — RESUMO POR REGRA (% de falha por regra por dia)
# ============================================================

df_resumo_regra = (df_logs
    .groupBy("tabela", "regra", "_data", "severidade")
    .agg(F.sum("qtd_registros_falhos").cast("long").alias("qtd_falhas"),
         F.sum("qtd_registros_total").cast("long").alias("qtd_total"))
    .withColumn("pct_falha", F.round(F.try_divide(F.col("qtd_falhas"), F.col("qtd_total")) * 100, 2).cast("double"))
    .select("tabela", "regra", "_data", "qtd_falhas", "qtd_total", "pct_falha", "severidade"))

print(f"Resumo por regra: {df_resumo_regra.count()} linhas (tabela × regra × dia)")
display(df_resumo_regra.orderBy("tabela", "_data", "regra"))

## Etapa 3 — Resumo por regra

Soma falhas e totais de todos os micro-lotes de cada dia, por tabela e regra, e calcula `pct_falha` com `try_divide` (sem erro de divisão por zero). Colunas e ordem iguais às das outras Golds da squad: `tabela`, `regra`, `_data`, `qtd_falhas`, `qtd_total`, `pct_falha`, `severidade`.

In [0]:
# ============================================================
# CÉLULA 4 — RESUMO POR TABELA (% de registros limpos por hora)
# ============================================================
# Calculado na Silver, linha a linha: uma linha é limpa se não falhou em
# nenhuma regra por linha. Regra nula (não avaliada) não conta como falha.

def resumo_por_tabela(tabela, cfg):
    df = spark.read.format("delta").options(**adls_options).load(cfg["silver"])
    falhou = reduce(lambda acc, c: acc | F.coalesce(F.col(c) == False, F.lit(False)),
                    cfg["regras_linha"], F.lit(False))
    return (df
        .withColumn("_data", data_do_lote(F.col("bronze_source_file")))
        .withColumn("_hora", hora_do_lote(F.col("bronze_source_file")))
        .withColumn("_limpo", ~falhou)
        .groupBy("_data", "_hora")
        .agg(F.sum(F.col("_limpo").cast("long")).alias("qtd_limpos"),
             F.count("*").cast("long").alias("qtd_total"))
        .withColumn("tabela", F.lit(tabela))
        .withColumn("pct_limpo", F.round(F.try_divide(F.col("qtd_limpos"), F.col("qtd_total")) * 100, 2).cast("double"))
        .select("_data", "_hora", "qtd_limpos", "qtd_total", "tabela", "pct_limpo"))

df_resumo_tabela = reduce(lambda a, b: a.unionByName(b),
                          [resumo_por_tabela(t, cfg) for t, cfg in TABELAS.items()])

sem_data = df_resumo_tabela.filter((F.col("_data") == "") | (F.col("_hora") == "")).count()
assert sem_data == 0, f"{sem_data} grupos com bronze_source_file fora do padrão real-time-data/AAAA/MM/DD/HHMMSS/"

print(f"Resumo por tabela: {df_resumo_tabela.count()} linhas (tabela × dia × hora)")
display(df_resumo_tabela.orderBy("tabela", "_data", "_hora"))

## Etapa 4 — Resumo por tabela

Para cada Silver, marca a linha como limpa quando nenhuma regra por linha deu `False` (o `coalesce` faz uma regra nula, não avaliada, não contar como falha) e agrega por `_data` e `_hora` do micro-lote. Assim `pct_limpo` é de fato o **% de registros que passaram em todas as regras**, e não uma média das taxas de falha. Colunas iguais às das outras Golds: `_data`, `_hora`, `qtd_limpos`, `qtd_total`, `tabela`, `pct_limpo`.

In [0]:
# ============================================================
# CÉLULA 5 — SINK 1: GRAVAR EM DELTA (ADLS), UM CAMINHO POR TABELA
# ============================================================

for tabela in TABELAS:
    for nome, df in [("dq_resumo_por_regra", df_resumo_regra), ("dq_resumo_por_tabela", df_resumo_tabela)]:
        destino = gold_path(tabela, nome)
        (df.filter(F.col("tabela") == tabela)
            .write.format("delta")
            .options(**adls_options)
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(destino))
        print(f"  [OK] {destino}")

## Etapa 5 — Sink 1: Delta no ADLS

Grava cada resumo em `squad1/gold/<tabela>/dq_resumo_por_regra` e `squad1/gold/<tabela>/dq_resumo_por_tabela`. O `overwrite` recalcula a Gold do zero a cada execução, mas atinge **somente os caminhos desta dupla**. Os caminhos genéricos `squad1/gold/gold_dq_resumo_por_*`, usados por outra dupla, não são mais tocados.

In [0]:
# ============================================================
# CÉLULA 6 — SINK 2: GRAVAR NO SQL SERVER E CONFERIR
# ============================================================

def opcoes_sql(nome_tabela):
    return {"host": sql_host, "port": "1433", "user": sql_user, "password": sql_password,
            "database": sql_database, "dbtable": nome_tabela}

for tabela, cfg in TABELAS.items():
    for prefixo, df in [("gold_dq_resumo_por_regra", df_resumo_regra), ("gold_dq_resumo_por_tabela", df_resumo_tabela)]:
        nome_sql = f"squad1.{prefixo}_{cfg['sufixo_sql']}"
        df_tab = df.filter(F.col("tabela") == tabela)
        df_tab.write.format("sqlserver").options(**opcoes_sql(nome_sql)).mode("overwrite").save()

        esperado = df_tab.count()
        gravado = spark.read.format("sqlserver").options(**opcoes_sql(nome_sql)).load().count()
        print(f"  {nome_sql}: esperado {esperado} | no SQL {gravado}")
        assert esperado == gravado, f"Divergência de contagem em {nome_sql}"

print("[OK] Gold gravada e conferida no SQL Server.")

## Etapa 6 — Sink 2: SQL Server

Grava as quatro tabelas no schema `squad1` com o conector nativo do Databricks (`format("sqlserver")`), em `overwrite`, e relê cada uma para conferir a contagem. Os nomes têm o sufixo da tabela (`_itens_pedido`, `_rastreamento_entregas`), no mesmo padrão das Golds das outras duplas, para não colidir com nenhuma tabela da squad.

As tabelas genéricas `squad1.gold_dq_resumo_por_regra` e `squad1.gold_dq_resumo_por_tabela`, criadas pela versão anterior deste notebook, **não são apagadas aqui**. Se o painel do Looker usar essas tabelas, ele deve passar a usar as novas, e a remoção das antigas deve ser combinada com a squad.